# Trie & Union Find — Cheat Sheet

**Companion to:** Trees & Graphs Study Guide (Union Find covered in Part 2 Type 5)

**What this adds:** Union Find consolidates the full implementation with all optimizations in one place. Trie was not covered in the study guides — this cheat sheet gives you the full template and problem patterns.

---
## When to Use

| Signal | Pattern to reach for |
| :--- | :--- |
| "autocomplete", "prefix search", "starts with" | Trie |
| "word search in a board", "dictionary words" | Trie |
| "count words with prefix", "wildcard matching" | Trie with extra fields |
| "connected components", "number of groups" | Union Find |
| "redundant connection", "cycle in undirected graph" | Union Find |
| "dynamic connectivity" — edges added one by one | Union Find |

---
## Patterns at a Glance

| Pattern | Key idea | Time | Problems |
| :--- | :--- | :--- | :--- |
| Trie insert / search | Each node = one character; children stored in dict or array | O(m) per op, m = word length | 208, 212 |
| Trie with prefix count | Track count at each node to answer prefix queries | O(m) per op | 1268, 745 |
| Union Find basic | `parent` array; `find` returns root | O(n) worst case without optimization | 547 |
| Union Find optimized | Path compression + union by rank | O(α(n)) ≈ O(1) per op | 547, 684, 128 |

---
# Part 0 — Quick Recap

**Trie** (prefix tree) is a tree where each path from root to a node spells out a prefix. It is the go-to structure whenever a problem involves searching, inserting, or matching strings by their prefixes. It was not covered in the study guides.

**Union Find** (Disjoint Set Union) tracks which nodes belong to the same connected component. It was covered in the Trees & Graphs study guide under Graphs Type 5. This cheat sheet consolidates the full implementation and application patterns in one place for quick reference.

**How they relate:** they don't share a mechanism or problem class. They are grouped here purely because both are compact, self-contained data structures with a small problem set — each fits in one section.

**Closest neighbors to distinguish from:**
- Trie vs Hashmap: use Trie when prefix queries matter; a hashmap can only do exact lookups
- Union Find vs BFS/DFS for connected components: use Union Find when edges are added dynamically; use BFS/DFS when the full graph is given upfront

---
# Part 1 — Trie

## Core idea

Each `TrieNode` holds a dictionary of children (one per character) and a flag marking whether it is the end of a valid word. Insert and search both walk the tree character by character. The root node is empty — it has no character of its own.

In [ ]:
# ── Core Trie Implementation (LC 208) ────────────────────────────────────────
class TrieNode:
    def __init__(self):
        self.children = {}              # char → TrieNode
        self.is_end   = False           # marks end of a valid word

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True

    def search(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children: return False
            node = node.children[ch]
        return node.is_end               # must end at a valid word node

    def startsWith(self, prefix):
        node = self.root
        for ch in prefix:
            if ch not in node.children: return False
            node = node.children[ch]
        return True                      # only need to reach end of prefix


# ── Trie with prefix count ────────────────────────────────────────────────────
# Add a count field to each node to track how many words pass through it
class TrieNodeWithCount:
    def __init__(self):
        self.children  = {}
        self.is_end    = False
        self.count     = 0              # number of words passing through this node

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNodeWithCount()
            node = node.children[ch]
            node.count += 1             # increment count at every node along the path
        node.is_end = True


# ── Trie + DFS for Word Search II (LC 212) ───────────────────────────────────
# Build a Trie from the word list, then DFS the board
# At each cell, check if current path exists in Trie before continuing
def findWords(board, words):
    trie = Trie()
    for w in words: trie.insert(w)

    res = set()
    rows, cols = len(board), len(board[0])

    def dfs(node, i, j, path):
        if node.is_end: res.add(path)
        tmp, board[i][j] = board[i][j], '#'     # mark visited
        for di, dj in [(0,1),(0,-1),(1,0),(-1,0)]:
            ni, nj = i+di, j+dj
            if 0<=ni<rows and 0<=nj<cols and board[ni][nj] in node.children:
                dfs(node.children[board[ni][nj]], ni, nj, path+board[ni][nj])
        board[i][j] = tmp                        # restore

    for i in range(rows):
        for j in range(cols):
            if board[i][j] in trie.root.children:
                dfs(trie.root.children[board[i][j]], i, j, board[i][j])
    return list(res)

## Variants

- **Array instead of dict for children:** use `[None] * 26` and `ord(ch) - ord('a')` as index — faster but only works for lowercase English letters
- **Wildcard search (`.` matches any char):** at `.` nodes, recurse into all existing children instead of looking up one key
- **Delete a word:** decrement counts on the way back; remove node if count hits 0

## Common mistakes

- Returning `True` in `search` when you reach end of word characters but `is_end` is `False` — the prefix exists but the full word doesn't
- Forgetting to restore the board cell after DFS in Word Search — always use a `tmp` variable
- Building Trie inside the search function on every call — build it once upfront

**Practice problems:**

| Problem | Pattern | Notes |
| :--- | :--- | :--- |
| LC 208 — Implement Trie | Core insert / search / startsWith | Foundation — implement from memory |
| LC 212 — Word Search II | Trie + DFS on board | Trie prunes paths that can't lead to valid words |
| LC 211 — Design Add and Search Words | Trie + wildcard DFS | At `.` recurse into all children |

---
# Part 2 — Union Find

## Core idea

Each node starts as its own parent. `find(x)` returns the root of x's component. `union(x, y)` merges two components by connecting their roots. Two nodes are in the same component if and only if `find(x) == find(y)`.

In [ ]:
# ── Full Union Find with both optimizations ───────────────────────────────────
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))    # each node is its own parent initially
        self.rank   = [0] * n           # rank = upper bound on tree height
        self.count  = n                 # number of connected components

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # path compression — flatten tree
        return self.parent[x]

    def union(self, x, y):
        px, py = self.find(x), self.find(y)
        if px == py: return False               # already in same component
        if self.rank[px] < self.rank[py]:       # union by rank — attach smaller under larger
            px, py = py, px
        self.parent[py] = px
        if self.rank[px] == self.rank[py]:
            self.rank[px] += 1
        self.count -= 1                         # one fewer component
        return True

    def connected(self, x, y):
        return self.find(x) == self.find(y)


# ── Application: Count Connected Components (LC 547) ─────────────────────────
def findCircleNum(isConnected):
    n  = len(isConnected)
    uf = UnionFind(n)
    for i in range(n):
        for j in range(i + 1, n):
            if isConnected[i][j]:
                uf.union(i, j)
    return uf.count                     # remaining components after all unions


# ── Application: Redundant Connection (LC 684) ───────────────────────────────
# The edge whose union() returns False creates a cycle
def findRedundantConnection(edges):
    uf = UnionFind(len(edges) + 1)
    for u, v in edges:
        if not uf.union(u, v):          # already connected — this edge is redundant
            return [u, v]

## Variants

- **Weighted Union Find:** store a `weight` array alongside `parent` to track relative values between nodes (LC 399)
- **Union Find on a grid:** map cell `(i, j)` to index `i * cols + j` to use a 1D Union Find on a 2D grid
- **Virtual node:** add a virtual node `n` and union all nodes of a group to it — useful when you need to check membership in a group quickly

## Common mistakes

- Initializing `parent` with size `n` when nodes are 1-indexed — use `n + 1` to avoid off-by-one
- Calling `find` without path compression — always use the recursive or iterative path compression version
- Forgetting that `union` returning `False` is meaningful — it means a cycle was detected

**Practice problems:**

| Problem | Pattern | Notes |
| :--- | :--- | :--- |
| LC 547 — Number of Provinces | Count components | Use `uf.count` after all unions |
| LC 684 — Redundant Connection | Cycle detection | Return edge where `union()` returns `False` |
| LC 128 — Longest Consecutive Sequence | Component size tracking | Track size array alongside parent |
| LC 721 — Accounts Merge | Union by email | Map emails to indices; union emails of same account |

---
# Decision Guide

```
Does the problem involve strings and prefix matching?
├── Yes → Trie
│         ├── Insert / search / startsWith only    → Core Trie (LC 208)
│         ├── Count words with a given prefix      → Trie with count field
│         ├── Wildcard search (. matches any char) → Trie + DFS at . nodes (LC 211)
│         └── Find all words on a board            → Trie + DFS on grid (LC 212)
│
└── Does the problem involve grouping nodes into components?
    ├── Yes → Union Find
    │         ├── Full graph given upfront          → BFS/DFS is simpler
    │         ├── Edges added dynamically           → Union Find
    │         ├── Detect cycle in undirected graph  → union() returns False
    │         ├── Count components                  → track uf.count
    │         └── Grid connectivity                 → map (i,j) → i*cols+j
    └── No  → Neither — reconsider Graph traversal or Sliding Window
```